# Математика на пальцах
### Вся статистика из ноутбука «Claude Orange» — простым языком, с примерами

Этот ноутбук объясняет **каждую формулу**, которую мы использовали в лабораторной «поймайте статистику, которая врёт».
Принцип один: сначала формула словами, потом крошечный пример, который можно посчитать на бумаге, а в конце — проверка через библиотеку, что мы не наврали.

**Содержание:**
1. Среднее, медиана, мода
2. Разброс: дисперсия и стандартное отклонение
3. Квартили и IQR
4. Перекос (skew)
5. Корреляция Пирсона `r` и `r²`
6. p-value: что это и чего это НЕ
7. z-score (стандартизация)
8. t-тест: сравниваем две группы
9. Cohen's d: размер эффекта
10. Парадокс Симпсона
11. p-hacking и множественные сравнения

In [1]:
import numpy as np
from scipy import stats

# Игрушечный набор: оценки 7 апельсинов по шкале 1-5 (с ним удобно считать руками)
x = np.array([2, 3, 3, 4, 4, 4, 12])  # 12 - намеренная "ошибка"/выброс, чтобы было видно как ломается среднее
print("Наши данные:", x)

Наши данные: [ 2  3  3  4  4  4 12]


## 1. Среднее, медиана, мода

**Среднее (mean)** — сложить все числа и разделить на количество.
$$\bar{x} = \frac{x_1 + x_2 + \dots + x_n}{n}$$

**Медиана** — число ровно посередине, если выстроить все по возрастанию. Половина значений меньше, половина больше.

**Мода** — самое частое значение.

👉 **Главная идея:** среднее «тянется» к выбросам, медиана — нет. Один аномально большой апельсин (12) поднимает среднее, но медиане всё равно.

In [2]:
vals = [2, 3, 3, 4, 4, 4, 5]   # 7 оценок по возрастанию
n = len(vals)

mean_hand = sum(vals) / n
median_hand = sorted(vals)[n // 2]      # средний элемент (n нечётное)
mode_hand = max(set(vals), key=vals.count)

print(f"Среднее  вручную = {mean_hand:.3f}")
print(f"Медиана  вручную = {median_hand}  (4-й элемент из 7)")
print(f"Мода     вручную = {mode_hand}  (чаще всего встречается)")

print("\nПроверка библиотекой:")
print(f"np.mean   = {np.mean(vals):.3f}")
print(f"np.median = {np.median(vals)}")
print(f"mode      = {stats.mode(vals, keepdims=False).mode}")

Среднее  вручную = 3.571
Медиана  вручную = 4  (4-й элемент из 7)
Мода     вручную = 4  (чаще всего встречается)

Проверка библиотекой:
np.mean   = 3.571
np.median = 4.0
mode      = 4


In [3]:
# Демонстрация: как ОДИН выброс врёт среднему
clean   = [2, 3, 3, 4, 4, 4, 5]
broken  = [2, 3, 3, 4, 4, 4, 12]   # последний апельсин - выброс
print(f"Без выброса: среднее={np.mean(clean):.2f}, медиана={np.median(clean):.1f}")
print(f"С выбросом : среднее={np.mean(broken):.2f}, медиана={np.median(broken):.1f}")
print("-> выброс поднял среднее на целый балл, а медиана не сдвинулась!")

Без выброса: среднее=3.57, медиана=4.0
С выбросом : среднее=4.57, медиана=4.0
-> выброс поднял среднее на целый балл, а медиана не сдвинулась!


**Вывод.** Когда среднее и медиана сильно расходятся — в данных есть перекос или выброс. Именно поэтому в Задаче 1 ноутбука мы смотрели на ОБА числа, а не только на среднее.

## 2. Разброс: дисперсия и стандартное отклонение

Среднее говорит «где центр», но не говорит «насколько данные разбросаны». Для этого считают **стандартное отклонение (std)**.

Шаги:
1. Для каждого числа берём отклонение от среднего: $(x_i - \bar{x})$
2. Возводим в квадрат (чтобы минусы не съели плюсы): $(x_i - \bar{x})^2$
3. Усредняем — это **дисперсия**: $\;s^2 = \frac{\sum (x_i - \bar{x})^2}{n-1}$
4. Берём корень — это **std**: $\;s = \sqrt{s^2}$

👉 std в тех же единицах, что и данные. Маленький std = все близко к среднему; большой = разброс широкий.

*(Делим на n−1, а не n — это поправка Бесселя для выборки. Для интуиции не важно.)*

In [4]:
data = [2, 4, 4, 4, 5, 5, 7]
mean = sum(data) / len(data)
print(f"Среднее = {mean:.3f}")

# Шаг за шагом
devs = [(v - mean) for v in data]
sq   = [(v - mean)**2 for v in data]
print("Отклонения от среднего :", [round(d,2) for d in devs])
print("Их квадраты            :", [round(s,2) for s in sq])

variance = sum(sq) / (len(data) - 1)   # дисперсия (выборочная)
std = variance ** 0.5
print(f"\nДисперсия вручную = {variance:.3f}")
print(f"Std       вручную = {std:.3f}")
print(f"Проверка np.std(ddof=1) = {np.std(data, ddof=1):.3f}")

Среднее = 4.429
Отклонения от среднего : [-2.43, -0.43, -0.43, -0.43, 0.57, 0.57, 2.57]
Их квадраты            : [5.9, 0.18, 0.18, 0.18, 0.33, 0.33, 6.61]

Дисперсия вручную = 2.286
Std       вручную = 1.512
Проверка np.std(ddof=1) = 1.512


**Вывод.** std ≈ 1.5 значит «типичный апельсин отклоняется от среднего примерно на 1.5». В Задаче 1 std спелости ≈ 1.2 при шкале 1–5 — умеренный разброс.

## 3. Квартили и IQR

Разрежем отсортированные данные на 4 равные части. Точки разреза — **квартили**:
- **Q1** (25%) — четверть значений ниже него
- **Q2** (50%) — это медиана
- **Q3** (75%) — три четверти значений ниже него

**IQR** (межквартильный размах) = Q3 − Q1 — это «ширина средних 50% данных».

👉 IQR используют для поиска выбросов: всё, что ниже `Q1 − 1.5·IQR` или выше `Q3 + 1.5·IQR`, — подозрительная точка («усы» боксплота).

In [5]:
d = np.array([1, 2, 3, 4, 5, 6, 7, 8, 100])  # 100 - явный выброс
q1 = np.quantile(d, 0.25)
q2 = np.quantile(d, 0.50)
q3 = np.quantile(d, 0.75)
iqr = q3 - q1
print(f"Q1 = {q1},  медиана(Q2) = {q2},  Q3 = {q3}")
print(f"IQR = Q3 - Q1 = {iqr}")

lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f"\nГраницы 'нормы': [{lo:.1f}, {hi:.1f}]")
outliers = d[(d < lo) | (d > hi)]
print(f"Выбросы (за границами): {outliers}  <- 100 пойман")

Q1 = 3.0,  медиана(Q2) = 5.0,  Q3 = 7.0
IQR = Q3 - Q1 = 4.0

Границы 'нормы': [-3.0, 13.0]
Выбросы (за границами): [100]  <- 100 пойман


## 4. Перекос (skewness)

**Skew** одним числом отвечает: куда «свешивается хвост» распределения?
- `skew ≈ 0` — симметрично (как колокол).
- `skew < 0` — **левый перекос**: длинный хвост влево, большинство значений справа. Тогда **mean < median**.
- `skew > 0` — **правый перекос**: хвост вправо. Тогда **mean > median**.

👉 Быстрый трюк без формулы: сравни среднее и медиану. Среднее меньше медианы → левый перекос (как `Ripeness` в Задаче 1).

In [6]:
left  = [1, 2, 5, 5, 5, 5, 5, 5]   # хвост влево
right = [5, 5, 5, 5, 5, 5, 8, 9]   # хвост вправо
for name, arr in [("Левый перекос", left), ("Правый перекос", right)]:
    print(f"{name:15s}: mean={np.mean(arr):.2f}, median={np.median(arr):.1f}, "
          f"skew={stats.skew(arr):+.2f}  ->  mean {'<' if np.mean(arr)<np.median(arr) else '>'} median")

Левый перекос  : mean=4.12, median=5.0, skew=-1.24  ->  mean < median
Правый перекос : mean=5.88, median=5.0, skew=+1.24  ->  mean > median


## 5. Корреляция Пирсона `r`

`r` отвечает: когда одна величина растёт, что делает другая? Это число от −1 до +1:
- **+1** — идеально растут вместе
- **0** — никакой линейной связи
- **−1** — одна растёт, другая идеально падает

Формула (по сути — «насколько согласованно две величины отклоняются от своих средних»):
$$r = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum (x_i-\bar{x})^2}\,\sqrt{\sum (y_i-\bar{y})^2}}$$

**`r²`** (просто r в квадрате) = какая доля разброса одной величины объясняется другой. r=0.63 → r²≈0.40 → «40% разброса объяснено».

⚠️ `r` ловит только **прямую линию**. Изогнутую связь (например, параболу) он не увидит. И главное: **корреляция ≠ причинность.**

In [7]:
# Почти идеальная связь: рост и вес
height = np.array([150, 160, 165, 170, 175, 180, 190])
weight = np.array([50,  60,  62,  68,  74,  80,  92])

r, p = stats.pearsonr(height, weight)
print(f"r   = {r:.3f}  (близко к +1 -> растут вместе)")
print(f"r^2 = {r**2:.3f}  -> {r**2*100:.0f}% разброса веса объяснено ростом")

# Контрпример: нелинейная связь, которую r НЕ видит
xx = np.array([-3,-2,-1,0,1,2,3])
yy = xx**2                      # идеальная парабола - связь ЕСТЬ
r2, _ = stats.pearsonr(xx, yy)
print(f"\nПарабола y=x^2: r = {r2:.3f}  <- r≈0, хотя связь идеальная! r видит только ПРЯМУЮ.")

r   = 0.994  (близко к +1 -> растут вместе)
r^2 = 0.989  -> 99% разброса веса объяснено ростом

Парабола y=x^2: r = 0.000  <- r≈0, хотя связь идеальная! r видит только ПРЯМУЮ.


## 6. p-value — самое непонятое число в статистике

Представь скептика, который говорит: **«Никакой связи нет, тебе просто повезло со случайными данными».** Эта позиция называется *нулевая гипотеза*.

**p-value = вероятность увидеть такой же сильный результат (или сильнее) ЧИСТО СЛУЧАЙНО, если скептик прав.**

- **маленькое p (< 0.05)** → «случайностью это объяснить трудно» → результат считаем значимым.
- **большое p** → «легко могло выйти случайно» → верить рано.

👉 Чего p-value НЕ говорит:
- НЕ говорит, что связь большая/важная (только что она «не случайна»).
- НЕ говорит вероятность, что гипотеза верна.
- p=0.04 и p=0.0001 — оба «значимы», но второй — гораздо увереннее.

In [8]:
# Смоделируем монетку: 10 бросков, выпало 8 орлов. Это перекос или везение?
# Скептик: монетка честная (p_орла = 0.5). Какова вероятность >=8 орлов случайно?
from scipy.stats import binomtest
res = binomtest(8, n=10, p=0.5, alternative='greater')
print(f"8 орлов из 10 при честной монетке: p-value = {res.pvalue:.3f}")
print("p > 0.05 -> 8 из 10 ещё может быть везением, монетку 'кривой' не объявляем")

res2 = binomtest(18, n=20, p=0.5, alternative='greater')
print(f"\nНо 18 орлов из 20: p-value = {res2.pvalue:.4f}")
print("p < 0.05 -> вот это уже трудно списать на случай")

8 орлов из 10 при честной монетке: p-value = 0.055
p > 0.05 -> 8 из 10 ещё может быть везением, монетку 'кривой' не объявляем

Но 18 орлов из 20: p-value = 0.0002
p < 0.05 -> вот это уже трудно списать на случай


## 7. z-score (стандартизация)

z-score отвечает: **на сколько стандартных отклонений данная точка отстоит от среднего?**
$$z = \frac{x - \bar{x}}{s}$$

- `z = 0` — точка ровно на среднем
- `z = +2` — на 2 std выше среднего
- `z = −1.5` — на 1.5 std ниже

👉 Правило: при нормальном распределении ~95% точек лежат в пределах |z| < 2, и ~99.7% — в пределах |z| < 3. Поэтому точки с **|z| > 2 или 3** — кандидаты в аномалии (как pH в Задаче 5).

In [9]:
vals = np.array([10, 12, 11, 13, 12, 11, 30])  # 30 - выброс
mean, std = vals.mean(), vals.std(ddof=1)
z = (vals - mean) / std
print(f"mean = {mean:.2f}, std = {std:.2f}\n")
for v, zz in zip(vals, z):
    flag = "  <- аномалия (|z|>2)" if abs(zz) > 2 else ""
    print(f"x={v:3d}  ->  z={zz:+.2f}{flag}")

mean = 14.14, std = 7.06

x= 10  ->  z=-0.59
x= 12  ->  z=-0.30
x= 11  ->  z=-0.45
x= 13  ->  z=-0.16
x= 12  ->  z=-0.30
x= 11  ->  z=-0.45
x= 30  ->  z=+2.25  <- аномалия (|z|>2)


## 8. t-тест — сравниваем средние двух групп

Вопрос: у группы A среднее 3.97, у группы B — 3.56. **Это реальная разница или просто шум выборки?**

t-тест считает число **t** = «насколько велика разница средних относительно разброса внутри групп»:
$$t \approx \frac{\text{разница средних}}{\text{шум (разброс / размер групп)}}$$

Большой |t| → разница заметна на фоне шума. По t и размеру групп считается **p-value** (см. раздел 6): мал ли шанс, что разница случайна.

👉 Здесь p-value снова отвечает на «не случайно ли?», но НЕ на «большая ли разница». Для размера разницы есть Cohen's d (раздел 9).

In [10]:
# Две группы оценок
group_A = np.array([4, 5, 4, 5, 4, 4, 5, 4, 5, 4])   # без дефектов
group_B = np.array([3, 4, 3, 4, 3, 3, 4, 2, 3, 4])   # с дефектами

t, p = stats.ttest_ind(group_A, group_B, equal_var=False)
print(f"Среднее A = {group_A.mean():.2f}")
print(f"Среднее B = {group_B.mean():.2f}")
print(f"Разница   = {group_A.mean()-group_B.mean():.2f}")
print(f"\nt = {t:.3f},  p = {p:.4f}")
print("p < 0.05 -> разница не случайна (как дефекты в Задаче 8)")

Среднее A = 4.40
Среднее B = 3.30
Разница   = 1.10

t = 4.093,  p = 0.0008
p < 0.05 -> разница не случайна (как дефекты в Задаче 8)


## 9. Cohen's d — а насколько разница БОЛЬШАЯ?

p-value говорит «разница есть». Cohen's d говорит **«насколько она велика»** — измеряет разницу средних в единицах стандартного отклонения:
$$d = \frac{\bar{x}_A - \bar{x}_B}{s_{\text{pooled}}}$$

Грубая шкала:
- `d ≈ 0.2` — **малый** эффект
- `d ≈ 0.5` — **средний**
- `d ≈ 0.8` — **большой**

👉 Зачем нужен: при больших выборках даже крошечная разница даёт p<0.05 («значимо»). Cohen's d не даст себя обмануть — покажет, что эффект на деле мелкий (как d=0.42 в Задаче 8).

In [11]:
def cohens_d(a, b):
    a, b = np.array(a), np.array(b)
    sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

d = cohens_d(group_A, group_B)
print(f"Cohen's d = {d:.3f}")
size = "большой" if abs(d)>=0.8 else "средний" if abs(d)>=0.5 else "малый"
print(f"Это {size} эффект")

Cohen's d = 1.831
Это большой эффект


## 10. Парадокс Симпсона

Связь, которая выглядит так в целом, **переворачивается внутри групп**. Звучит невозможно — но вот живой пример.

Идея: если группы сильно различаются и по X, и по Y, то «общая» связь — артефакт смешивания, а не реальная закономерность.

In [12]:
# Две группы. Внутри КАЖДОЙ: больше часов учёбы -> выше балл (положительная связь).
# Группа 1 (мало учится, легкий предмет):
g1_hours = np.array([1, 2, 3, 4])
g1_score = np.array([60, 62, 64, 66])    # растёт с часами
# Группа 2 (много учится, сложный предмет, баллы ниже):
g2_hours = np.array([6, 7, 8, 9])
g2_score = np.array([50, 52, 54, 56])    # тоже растёт с часами

# Внутри групп связь положительная:
print("Внутри группы 1: r =", round(stats.pearsonr(g1_hours, g1_score)[0], 2), "(больше учишься -> выше балл)")
print("Внутри группы 2: r =", round(stats.pearsonr(g2_hours, g2_score)[0], 2), "(тоже!)")

# А теперь смешаем и посчитаем ОБЩУЮ связь:
all_h = np.concatenate([g1_hours, g2_hours])
all_s = np.concatenate([g1_score, g2_score])
r_all = stats.pearsonr(all_h, all_s)[0]
print(f"\nНо ОБЩАЯ связь по всем данным: r = {r_all:.2f}  <- ОТРИЦАТЕЛЬНАЯ!")
print("Вывод 'больше учишься -> ниже балл' - ложь от смешивания групп. Это парадокс Симпсона.")

Внутри группы 1: r = 1.0 (больше учишься -> выше балл)
Внутри группы 2: r = 1.0 (тоже!)

Но ОБЩАЯ связь по всем данным: r = -0.67  <- ОТРИЦАТЕЛЬНАЯ!
Вывод 'больше учишься -> ниже балл' - ложь от смешивания групп. Это парадокс Симпсона.


**Как в ноутбуке.** В Задаче 6: `Size↔Quality` в целом −0.24, но внутри цвета Orange +0.33. Всегда спрашивай: а нет ли внутри скрытых групп?

## 11. p-hacking и множественные сравнения

Самая коварная ловушка дня. Правило простое:

> Если порог значимости p < 0.05, то даже у **чистого шума** примерно **1 из 20** проверок пройдёт «значимость» случайно.

Проверишь 20 гипотез — в среднем 1 «открытие» гарантировано, хотя там пусто. Проверишь 780 пар (как в охоте на призрак) — получишь ~39 «значимых» случайностей.

👉 Это и есть **p-hacking**: перебрать кучу вариантов и показать только тот, что «выстрелил». Так рождаются ложные открытия и так переобучаются модели.

**Защита:** фиксируй гипотезу заранее; делай поправку на число проверок (напр. Бонферрони: порог 0.05/число тестов); проверяй находку на новых данных.

In [13]:
# Сгенерируем 200 пар ЧИСТОГО шума и посчитаем, сколько 'значимы' случайно
rng = np.random.default_rng(0)
n_tests = 200
false_hits = 0
for _ in range(n_tests):
    a = rng.normal(size=40)
    b = rng.normal(size=40)          # a и b независимы по построению - связи НЕТ
    _, p = stats.pearsonr(a, b)
    if p < 0.05:
        false_hits += 1

print(f"Проверили {n_tests} пар чистого шума (настоящей связи нет ни в одной).")
print(f"'Значимыми' (p<0.05) оказались: {false_hits}")
print(f"Ожидалось случайно: ~{0.05*n_tests:.0f}")
print("\n-> вот откуда берутся 'впечатляющие' корреляции из ничего. Цифра убеждает, но лжёт.")

Проверили 200 пар чистого шума (настоящей связи нет ни в одной).
'Значимыми' (p<0.05) оказались: 17
Ожидалось случайно: ~10

-> вот откуда берутся 'впечатляющие' корреляции из ничего. Цифра убеждает, но лжёт.


---
## Шпаргалка

| Понятие | Отвечает на вопрос | Тревожный сигнал |
|---|---|---|
| **Среднее** | где центр? | сильно ≠ медианы → перекос/выброс |
| **Медиана** | где центр, устойчиво к выбросам? | — |
| **Std** | насколько разбросано? | большой → данные «гуляют» |
| **Квартили / IQR** | где средние 50%? | точки за 1.5·IQR → выбросы |
| **Skew** | куда свешивается хвост? | \|skew\| велик → не симметрично |
| **Pearson r** | растут ли вместе (линейно)? | путают с причинностью |
| **r²** | какую долю разброса объясняет? | высокий ≠ правда (см. p-hacking) |
| **p-value** | не случайно ли? | мал ≠ «большой эффект» |
| **z-score** | сколько std от среднего? | \|z\|>3 → аномалия |
| **t-тест** | различаются ли средние групп? | значимо ≠ важно |
| **Cohen's d** | насколько велик эффект? | мал → разница неинтересна |
| **Симпсон** | не переворачивается ли связь в группах? | смешаны разные группы |
| **p-hacking** | не перебрали ли мы лишнего? | много тестов → ложные «открытия» |

*Формулы считает машина. За нами остаётся скепсис.*